# 📧 Spam Detection System Portfolio Project

Welcome to the portfolio project for **Naive Bayes**. In this notebook, we will tackle the quintessential NLP classification problem: Spam Detection.

## 1. Business Problem
An email provider needs to filter incoming messages into the Inbox or the Spam folder. The primary constraint is that **False Positives (flagging a real, important email as spam) are highly destructive** to user trust. We need a fast, probabilistic model that heavily prioritizes High Precision.

## 2. Dataset Description
We will simulate a text corpus of emails.

**Features:**
- `EmailText`: The raw string contents of the email.
- **Target:** `Label` (1 for Spam, 0 for Ham/Not Spam).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

spam_emails = [
    "WINNER! You have won a free lottery ticket. Click here to claim your prize!",
    "URGENT: Your bank account has been suspended. Please reply with your password.",
    "Get cheap viagra and medications now! Limited time offer.",
    "Congratulations! You are the 1000th visitor. Claim your free iPad now.",
    "Make $5000 a week working from home! No experience needed. Free money."
] * 200

ham_emails = [
    "Hey, are we still meeting for lunch tomorrow at 12?",
    "Please find attached the quarterly financial report you requested.",
    "Can you review my pull request? I fixed the bug in the login module.",
    "Mom, I will be home late tonight. Don't wait up for me.",
    "Reminder: Dental appointment on Thursday at 9 AM."
] * 800

data = spam_emails + ham_emails
labels = [1] * len(spam_emails) + [0] * len(ham_emails)

df = pd.DataFrame({'EmailText': data, 'Label': labels})
df = df.sample(frac=1).reset_index(drop=True) # Shuffle

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df['MessageLength'] = df['EmailText'].apply(len)

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='MessageLength', hue='Label', element="step")
plt.title('Email Length Distribution (Spam vs Ham)')
plt.show()

## 4. Feature Engineering
We need to convert raw text into a matrix of word counts using `CountVectorizer`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

X = df['EmailText']
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))

## 5. Model Training & 6. Hyperparameter Tuning
We use Multinomial Naive Bayes. The main hyperparameter is `alpha` (Laplace Smoothing).

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.1, 0.5, 1.0, 5.0, 10.0]}
grid = GridSearchCV(MultinomialNB(), param_grid, cv=5, scoring='precision') # Optimize for Precision!
grid.fit(X_train_vec, y_train)

best_nb = grid.best_estimator_
print("Best Alpha:", grid.best_params_['alpha'])

## 7. Evaluation & 8. Error Analysis
Did we achieve 100% precision?

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_nb.predict(X_test_vec)

print(classification_report(y_test, y_pred))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Reds')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("Error Analysis:")
print("We want the top-right cell (False Positives) to be absolutely 0.")

## 9. Visualizations & 10. Business Insights

In [ ]:
feature_names = vectorizer.get_feature_names_out()

# Get log probabilities of words given spam
spam_log_probs = best_nb.feature_log_prob_[1]
top_spam_indices = spam_log_probs.argsort()[::-1][:10]

top_spam_words = [feature_names[i] for i in top_spam_indices]

plt.figure(figsize=(8, 4))
sns.barplot(x=spam_log_probs[top_spam_indices], y=top_spam_words, orient='h')
plt.title("Top 10 Most 'Spammy' Words (Log Probability)")
plt.xlabel("Log Probability")
plt.show()

print("Business Insights:")
print("- Words like 'free', 'winner', and 'urgent' heavily trigger the spam filter.")
print("- The Naive Bayes assumption (that words are independent) works phenomenally well for basic text classification.")

## 11. Future Improvements
- Implement TF-IDF instead of basic CountVectorizer.
- Add N-grams (bi-grams like 'free money') to capture local context.